In [1]:
# Cell 1 — Install and import everything we need
!pip install kagglehub snowflake-connector-python pandas -q

import kagglehub
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.7/81.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.7 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 26.0.0 which is inco

In [2]:
# Cell 2 — Download dataset
path = kagglehub.dataset_download("elemento/nyc-yellow-taxi-trip-data")
print("Path to dataset files:", path)

import os
files = os.listdir(path)
print("Files available:", files)

100%|██████████| 1.78G/1.78G [00:16<00:00, 113MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/elemento/nyc-yellow-taxi-trip-data/versions/2
Files available: ['yellow_tripdata_2016-01.csv', 'yellow_tripdata_2016-02.csv', 'yellow_tripdata_2016-03.csv', 'yellow_tripdata_2015-01.csv']


In [3]:
# Cell 3 — Load one file into a dataframe
import os

file_path = os.path.join(path, 'yellow_tripdata_2016-01.csv')

print("Loading data... this may take 30 seconds ⏳")
df = pd.read_csv(file_path)

print(f"✅ Loaded successfully!")
print(f"📊 Total rows: {len(df):,}")
print(f"📋 Total columns: {len(df.columns)}")
print(f"\nColumn names:")
print(df.columns.tolist())

Loading data... this may take 30 seconds ⏳
✅ Loaded successfully!
📊 Total rows: 10,906,858
📋 Total columns: 19

Column names:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount']


In [4]:
# Cell 4 — Quick look at the data
print("First 3 rows:")
print(df.head(3))

print("\n📊 Basic stats:")
print(f"Date range: {df['tpep_pickup_datetime'].min()} to {df['tpep_pickup_datetime'].max()}")
print(f"Avg fare: ${df['fare_amount'].mean():.2f}")
print(f"Avg trip distance: {df['trip_distance'].mean():.2f} miles")
print(f"Avg passengers: {df['passenger_count'].mean():.2f}")

print("\n❓ Missing values per column:")
print(df.isnull().sum())

First 3 rows:
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         2  2016-01-01 00:00:00   2016-01-01 00:00:00                2   
1         2  2016-01-01 00:00:00   2016-01-01 00:00:00                5   
2         2  2016-01-01 00:00:00   2016-01-01 00:00:00                1   

   trip_distance  pickup_longitude  pickup_latitude  RatecodeID  \
0           1.10        -73.990372        40.734695           1   
1           4.90        -73.980782        40.729912           1   
2          10.54        -73.984550        40.679565           1   

  store_and_fwd_flag  dropoff_longitude  dropoff_latitude  payment_type  \
0                  N         -73.981842         40.732407             2   
1                  N         -73.944473         40.716679             1   
2                  N         -73.950272         40.788925             1   

   fare_amount  extra  mta_tax  tip_amount  tolls_amount  \
0          7.5    0.5      0.5         0.0           0.

In [5]:
# Cell 5 — Clean the data
print("Cleaning data")

# Keep only good rows
df_clean = df[
    (df['fare_amount'] > 0) &          # no zero fares
    (df['trip_distance'] > 0) &        # no zero distance
    (df['passenger_count'] > 0) &      # no empty trips
    (df['total_amount'] > 0)           # no zero totals
].copy()

# Clean column names — Snowflake prefers uppercase
df_clean.columns = [col.upper() for col in df_clean.columns]

# Convert datetime columns
df_clean['TPEP_PICKUP_DATETIME'] = pd.to_datetime(df_clean['TPEP_PICKUP_DATETIME'])
df_clean['TPEP_DROPOFF_DATETIME'] = pd.to_datetime(df_clean['TPEP_DROPOFF_DATETIME'])

print(f"Rows before cleaning: {len(df):,}")
print(f"Rows after cleaning:  {len(df_clean):,}")
print(f"Removed: {len(df) - len(df_clean):,} bad rows")

Cleaning data
Rows before cleaning: 10,906,858
Rows after cleaning:  10,837,367
Removed: 69,491 bad rows


In [10]:
conn = snowflake.connector.connect(
    user="YOUR_SNOWFLAKE_USERNAME",
    password="YOUR_SNOWFLAKE_PASSWORD",
    account="YOUR_ACCOUNT_IDENTIFIER",
    warehouse="MY_WH",
    database="PIPELINE_DB",
    schema="RAW"
)

print("✅ Connected to Snowflake successfully!")

✅ Connected to Snowflake successfully!


In [14]:
#Fix datetime columns
df_clean['TPEP_PICKUP_DATETIME'] = df_clean['TPEP_PICKUP_DATETIME'].astype(str)
df_clean['TPEP_DROPOFF_DATETIME'] = df_clean['TPEP_DROPOFF_DATETIME'].astype(str)

print("✅ Datetime columns converted")
print("Sample pickup value:", df_clean['TPEP_PICKUP_DATETIME'].iloc[0])
print("Sample dropoff value:", df_clean['TPEP_DROPOFF_DATETIME'].iloc[0])

✅ Datetime columns converted
Sample pickup value: 2016-01-01 00:00:00
Sample dropoff value: 2016-01-01 00:00:00


In [12]:
#Load data into Snowflake
print("⏳ Loading 10.8M rows into Snowflake... grab a coffee! ☕")

success, num_chunks, num_rows, output = write_pandas(
    conn=conn,
    df=df_clean,
    table_name="TAXI_TRIPS",
    database="PIPELINE_DB",
    schema="RAW"
)

print(f"✅ Success: {success}")
print(f"📦 Chunks uploaded: {num_chunks}")
print(f"📊 Rows loaded: {num_rows:,}")

⏳ Loading 10.8M rows into Snowflake... grab a coffee! ☕


/tmp/ipykernel_2447/2334272585.py:4: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, num_chunks, num_rows, output = write_pandas(


✅ Success: True
📦 Chunks uploaded: 1
📊 Rows loaded: 10,837,367
